# Lección 2 — Chains Avanzados y Patrones

## Módulo 2 — Chains / Cadenas | Semana 4

En la lección anterior construimos chains secuenciales: paso 1 → paso 2 → paso 3. Eso es suficiente para muchos casos, pero en producción te vas a encontrar con tres problemas que los chains lineales no resuelven:

1. **Necesitas ejecutar chains diferentes** según el tipo de input (un bug report no se procesa igual que un feature request).
2. **Necesitas velocidad** — si 4 tareas son independientes, ¿por qué esperar a que terminen una por una?
3. **Necesitas procesar documentos demasiado grandes** para una sola llamada al LLM.

En esta lección vamos a cubrir los patrones avanzados que resuelven cada uno de estos problemas:

| Sección | Patrón | Problema que resuelve |
|---------|--------|-----------------------|
| 4.1 | **Router Chains** | Dirigir el input al chain correcto según su tipo |
| 4.2 | **Parallel Chains** | Ejecutar tareas independientes simultáneamente |
| 4.3 | **Map-Reduce** | Procesar documentos que exceden la ventana de contexto |
| 4.4 | **LangChain** | Referencia al framework más popular (y por qué no lo usamos) |

> **Contexto del ejemplo**: Seguimos trabajando con **Klimbook**, la app de escalada. Imagina que estás construyendo el sistema de soporte al usuario y el pipeline de release notes.

---

## 4.1 Branching / Router Chains

### ¿Qué es un Router Chain?

Un **router chain** es el equivalente a un `switch/case` pero potenciado por un LLM. En lugar de escribir reglas fijas con `if/elif`, dejas que el modelo **clasifique** el input y luego lo **despacha** al chain especializado correspondiente.

### ¿Por qué no usar if/elif con keywords?

Podrías intentar algo como:

```python
if "error" in message or "bug" in message:
    handle_bug(message)
elif "quiero" in message or "sería bueno" in message:
    handle_feature(message)
```

Pero esto falla rápido. Un usuario podría escribir *"La pantalla se queda en blanco cuando intento guardar mi ruta"* — es claramente un bug, pero no contiene la palabra "error" ni "bug". Un LLM entiende la **intención**, no solo las palabras.

### El patrón visual

```
Input del usuario
    ↓
┌───────────────────┐
│   Clasificador    │  ← Claude decide la categoría
└────────┬──────────┘
         │
    ┌────┼────┬────────┐
    ↓    ↓    ↓        ↓
  Bug  Feature Question  Feedback
 Chain  Chain   Chain    Chain
    ↓    ↓    ↓        ↓
 Output Output Output  Output
```

### Ejemplo completo: Sistema de soporte de Klimbook

Imagina que **Klimbook** recibe mensajes de usuarios a través de un formulario de soporte. Cada mensaje necesita un tratamiento diferente — un bug report debe convertirse en un issue de GitHub, un feature request debe evaluarse, una pregunta debe responderse, etc.

### Paso 0 — Setup e imports

Antes de empezar, necesitamos el cliente de Anthropic y las configuraciones que venimos usando. Si ya hiciste la lección anterior, esto te será familiar:

In [ ]:
import asyncio
import json
import logging
from anthropic import AsyncAnthropic

# Configuración de logging para ver el flujo del chain
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# Cliente async de Anthropic
client = AsyncAnthropic()  # usa ANTHROPIC_API_KEY del environment

# Modelos disponibles — misma convención de lecciones anteriores
MODELS = {
    "haiku": {"id": "claude-3-5-haiku-20241022"},   # Rápido y barato → clasificación, tareas simples
    "sonnet": {"id": "claude-sonnet-4-20250514"},  # Balance → análisis, generación de contenido
}

### Paso 1 — Clasificar el mensaje

El primer paso del router es **clasificar** el input. Para esto usamos **Haiku** con `temperature=0` porque:

- Es una tarea de **clasificación**, no de generación creativa — necesitas consistencia.
- `max_tokens=50` porque solo esperamos **una palabra** como respuesta.
- Haiku es ~10x más barato que Sonnet, y para clasificación es igual de preciso.

> **Regla general**: si la tarea tiene una respuesta "correcta" única (clasificación, extracción, formato), usa `temperature=0`. Si necesitas variedad o creatividad, sube la temperatura.

In [ ]:
# Categorías válidas para el router
VALID_CATEGORIES = {"bug_report", "feature_request", "question", "feedback"}

async def step_classify(message: str) -> str:
    """Clasifica un mensaje de usuario de Klimbook en una categoría."""
    response = await client.messages.create(
        model=MODELS["haiku"]["id"],
        system="""Eres el clasificador de mensajes de soporte de Klimbook (app de escalada).
                Clasifica el mensaje del usuario en UNA de estas categorías:
                - bug_report: reporta un error, crash, o mal funcionamiento
                - feature_request: pide una nueva funcionalidad o mejora
                - question: hace una pregunta sobre cómo usar la app
                - feedback: da una opinión, comentario general, o agradecimiento

                Responde SOLO con la categoría, sin explicación ni puntuación.""",
        messages=[{"role": "user", "content": message}],
        temperature=0.0,
        max_tokens=50,
    )
    category = response.content[0].text.strip().lower()

    # Validación: si el modelo devuelve algo inesperado, fallback a feedback
    if category not in VALID_CATEGORIES:
        logger.warning(f"Categoría inesperada: '{category}' → fallback a 'feedback'")
        return "feedback"

    return category


# Probemos con un mensaje de ejemplo
test_msg = "La app se crashea cuando intento guardar una ruta con más de 50 waypoints"
category = await step_classify(test_msg)
print(f"Mensaje: {test_msg}")
print(f"Categoría: {category}")  # Esperamos: bug_report

Observaciones del clasificador:

- **Validación del output**: siempre valida lo que devuelve el LLM. Aunque le digas "responde SOLO con X", a veces agrega un punto, un espacio, o una explicación. El `.strip().lower()` y la validación contra `VALID_CATEGORIES` cubren estos casos.
- **Fallback explícito**: si el modelo devuelve algo inesperado (y va a pasar), en lugar de crashear, redirigimos a `feedback` que es la categoría más segura.

### Paso 2 — Chains especializados por categoría

Ahora definimos **un chain diferente para cada categoría**. Cada uno tiene su propio system prompt, modelo, y temperatura optimizados para la tarea:

| Categoría | Modelo | Temperature | ¿Por qué? |
|-----------|--------|-------------|------------|
| `bug_report` | Sonnet | 0.3 | Análisis técnico complejo, necesita precisión |
| `feature_request` | Sonnet | 0.3 | Evaluación de viabilidad, requiere razonamiento |
| `question` | Sonnet | 0.5 | Respuesta informativa, algo de naturalidad |
| `feedback` | Haiku | 0.7 | Respuesta simple y amigable, más creatividad |

> **Patrón clave**: tareas analíticas → modelos grandes y temperatura baja. Tareas conversacionales → modelos rápidos y temperatura más alta.

In [ ]:
async def handle_bug(message: str) -> str:
    """Chain para bug reports: genera un draft de issue para GitHub."""
    response = await client.messages.create(
        model=MODELS["sonnet"]["id"],
        system="""Eres un QA engineer senior de Klimbook (app de escalada para iOS/Android).
        Analiza el bug report del usuario y genera un draft de issue para GitHub con este formato:

        ## Título
        (título conciso y descriptivo)

        ## Descripción
        (resumen del problema)

        ## Pasos para reproducir
        1. ...
        2. ...

        ## Comportamiento esperado
        ...

        ## Comportamiento actual
        ...

        ## Severidad
        (low / medium / high / critical)

        ## Componente afectado
        (UI / Backend / Database / Auth / Maps / etc.)""",
        messages=[{"role": "user", "content": message}],
        temperature=0.3,
        max_tokens=1000,
    )
    return response.content[0].text


async def handle_feature(message: str) -> str:
    """Chain para feature requests: evalúa viabilidad y genera draft."""
    response = await client.messages.create(
        model=MODELS["sonnet"]["id"],
        system="""Eres un product manager técnico de Klimbook (app de escalada).
        Analiza el feature request del usuario y genera un draft con:

        ## Título descriptivo
        ...

        ## Caso de uso
        (¿Quién lo usaría? ¿En qué situación?)

        ## Descripción de la funcionalidad
        (¿Qué debería hacer exactamente?)

        ## Viabilidad técnica
        (Alta / Media / Baja — con justificación breve)

        ## Complejidad estimada
        (S / M / L / XL)

        ## Labels sugeridos
        (enhancement, UX, backend, etc.)""",
        messages=[{"role": "user", "content": message}],
        temperature=0.3,
        max_tokens=1000,
    )
    return response.content[0].text


async def handle_question(message: str) -> str:
    """Chain para preguntas: responde basándose en el conocimiento de la app."""
    response = await client.messages.create(
        model=MODELS["sonnet"]["id"],
        system="""Eres un agente de soporte de Klimbook, una app de escalada que permite:
        - Buscar y guardar rutas de escalada con fotos y grados
        - Trackear tu progreso y estadísticas (rutas completadas, grado máximo, etc.)
        - Crear y compartir listas de rutas (wishlist, proyectos, etc.)
        - Seguir a otros escaladores y ver su actividad

        Responde la pregunta del usuario de forma clara, amigable y concisa.
        Si no sabes la respuesta, sé honesto y sugiere contactar soporte por email.""",
        messages=[{"role": "user", "content": message}],
        temperature=0.5,
        max_tokens=500,
    )
    return response.content[0].text


async def handle_feedback(message: str) -> str:
    """Chain para feedback: agradece y registra el sentimiento."""
    response = await client.messages.create(
        model=MODELS["haiku"]["id"],
        system="""Eres el community manager de Klimbook (app de escalada).
        Agradece el feedback del usuario de forma genuina y breve (2-3 oraciones).
        Si el feedback es negativo, muestra empatía y asegura que el equipo lo revisará.
        Si es positivo, compártelo con entusiasmo.""",
        messages=[{"role": "user", "content": message}],
        temperature=0.7,
        max_tokens=200,
    )
    return response.content[0].text

**¿Por qué cada handler tiene configuración diferente?**

Fíjate en las decisiones de diseño:

- **`handle_bug`** usa Sonnet + temp 0.3 → necesita razonamiento estructurado para inferir severidad, pasos de reproducción, y componentes afectados a partir de un mensaje informal del usuario.
- **`handle_feedback`** usa Haiku + temp 0.7 → solo necesita ser amigable. No vale la pena pagar Sonnet para decir "gracias por tu comentario".
- Los `max_tokens` también varían: un bug report genera ~500 tokens (issue completo), un agradecimiento ~50 tokens.

> **Costo en producción**: si recibes 1,000 mensajes/día y el 40% son feedback, estás ahorrando ~60% del costo en esos mensajes al usar Haiku en lugar de Sonnet.

### Paso 3 — El router que conecta todo

El router es simplemente un **diccionario** que mapea categorías a funciones handlers, más una función orquestadora que llama al clasificador y despacha:

In [ ]:
# Mapeo categoría → handler
ROUTES: dict[str, callable] = {
    "bug_report": handle_bug,
    "feature_request": handle_feature,
    "question": handle_question,
    "feedback": handle_feedback,
}

async def router(message: str) -> dict[str, str]:
    """Router principal: clasifica el mensaje y lo despacha al chain correcto."""
    # Paso 1: Clasificar
    category = await step_classify(message)
    logger.info(f"[Router] Mensaje clasificado como: {category}")

    # Paso 2: Obtener el handler (con fallback)
    handler = ROUTES.get(category, handle_feedback)

    # Paso 3: Ejecutar el chain especializado
    result = await handler(message)

    return {
        "category": category,
        "response": result,
    }

Probemos el router con diferentes tipos de mensajes de usuarios reales de Klimbook:

In [ ]:
# Mensajes de prueba — cada uno debería ir a un chain diferente
test_messages = [
    "La app se cierra sola cuando intento subir una foto de más de 10MB a mi perfil",
    "Estaría genial poder comparar mi progreso con amigos en una tabla de líderes",
    "¿Cómo puedo exportar mis rutas guardadas a un archivo GPX?",
    "Llevo usando Klimbook 6 meses y me encanta, es la mejor app de escalada!",
]

for msg in test_messages:
    result = await router(msg)
    print(f"{'─' * 60}")
    print(f" Mensaje: {msg}")
    print(f"  Categoría: {result['category']}")
    print(f" Respuesta:\n{result['response'][:200]}...")
    print()

### Mejora: Confidence Thresholds

¿Qué pasa si un usuario escribe algo ambiguo como *"Interesante la app"*? ¿Es feedback positivo o una pregunta sarcástica? El clasificador va a responder algo, pero podría no estar seguro.

**Solución**: pedirle al clasificador que devuelva también un **score de confianza**. Si está por debajo de un umbral, en lugar de adivinar, pedimos **clarificación** al usuario.

> **Truco del `assistant` prefill**: fíjate que pasamos `{"role": "assistant", "content": "{"}` en los messages. Esto le dice a Claude que ya empezó a escribir JSON, así que continuará con la estructura JSON en lugar de escribir texto libre. Es una técnica muy útil para forzar formato.

In [ ]:
async def step_classify_with_confidence(message: str) -> tuple[str, float]:
    """Clasifica un mensaje y devuelve un score de confianza (0.0 - 1.0)."""
    response = await client.messages.create(
        model=MODELS["haiku"]["id"],
        system="""Clasifica el mensaje de soporte de Klimbook y da un score de confianza.
        Categorías válidas: bug_report, feature_request, question, feedback.
        Responde SOLO con JSON: {"category": "...", "confidence": 0.0-1.0}""",
        messages=[
            {"role": "user", "content": message},
            {"role": "assistant", "content": "{"},  # Prefill: fuerza formato JSON
        ],
        temperature=0.0,
        max_tokens=100,
    )
    # El modelo continuó desde "{", así que reconstituimos el JSON completo
    parsed = json.loads("{" + response.content[0].text)
    return parsed["category"], parsed["confidence"]


async def router_with_confidence(message: str, threshold: float = 0.7) -> dict:
    """Router que pide clarificación si la confianza es baja."""
    category, confidence = await step_classify_with_confidence(message)

    if confidence < threshold:
        logger.warning(f"[Router] Baja confianza ({confidence:.0%}) para: '{message[:50]}...'")
        return {
            "category": "unclear",
            "confidence": confidence,
            "response": (
                "No estoy seguro de entender tu mensaje. "
                "¿Podrías darme más detalles? Por ejemplo:\n"
                "- Si es un **error**, describe qué pasó y qué esperabas.\n"
                "- Si es una **sugerencia**, cuéntame qué te gustaría ver.\n"
                "- Si es una **pregunta**, reformúlala con más contexto."
            ),
        }

    handler = ROUTES.get(category, handle_feedback)
    result = await handler(message)

    return {
        "category": category,
        "confidence": confidence,
        "response": result,
    }


# Probemos con un mensaje ambiguo vs uno claro
messages = [
    "Interesante",                              # Ambiguo — baja confianza esperada
    "No puedo iniciar sesión desde ayer, me sale error 500",  # Claro — bug_report
]

for msg in messages:
    result = await router_with_confidence(msg)
    print(f" '{msg}'")
    print(f"   → {result['category']} (confianza: {result['confidence']:.0%})")
    print()

---

## 4.2 Parallel Chains con `asyncio`

### El problema: chains secuenciales son lentos cuando no necesitan serlo

Hasta ahora todas nuestras llamadas al LLM han sido **secuenciales** — paso 1 termina, empieza paso 2, termina, empieza paso 3. Pero ¿qué pasa cuando varios pasos son **independientes** entre sí?

**Ejemplo concreto**: en el pipeline de Release Notes de Klimbook, una vez que tienes el markdown de las notas, necesitas generar versiones para 4 plataformas diferentes. Cada una es independiente — no necesitas esperar a que termine la versión de Play Store para empezar la de App Store.

### Visualización: secuencial vs paralelo

```
                    ┌→ GitHub     (2.1s) ──┐
                    │                      │
Release Notes MD ───┼→ Play Store (1.8s) ──┼→ Resultados
                    │                      │
                    ├→ App Store  (2.3s) ──┤
                    │                      │
                    └→ Ko-fi      (1.5s) ──┘
```

| Modo | Tiempo total | Cálculo |
|------|-------------|---------|
| **Secuencial** | 7.7s | 2.1 + 1.8 + 2.3 + 1.5 |
| **Paralelo** | 2.3s | max(2.1, 1.8, 2.3, 1.5) |

**~3.3x más rápido** sin ningún esfuerzo extra en la lógica. La mejora es mayor cuanto más tareas tengas.

> **¿Cuándo paralelizar?** Cuando las tareas **no dependen** una de otra. Si el resultado de la tarea A es input de la tarea B, NO puedes paralelizarlas.

### `asyncio.gather` — ejecutar todo en paralelo y esperar a todos

`asyncio.gather()` es la forma más sencilla: lanza todas las coroutines simultáneamente y espera a que **TODAS** terminen. El tiempo total es el del más lento, no la suma.

Primero, definamos las funciones de formateo para cada plataforma:

In [ ]:
import time

async def format_for_platform(platform: str, notes_md: str) -> str:
    """Formatea release notes para una plataforma específica."""
    platform_prompts = {
        "github": "Genera release notes para GitHub en markdown con secciones: New Features, Bug Fixes, Breaking Changes.",
        "playstore": "Genera la descripción de actualización para Google Play Store. Máximo 500 caracteres, tono casual y amigable.",
        "appstore": "Genera las release notes para App Store de Apple. Máximo 4000 caracteres, tono profesional.",
        "kofi": "Genera un post de actualización para Ko-fi (plataforma de supporters). Tono personal y agradecido, incluye emojis.",
    }

    start = time.time()
    response = await client.messages.create(
        model=MODELS["haiku"]["id"],
        system=platform_prompts[platform],
        messages=[{"role": "user", "content": f"Release notes originales:\n\n{notes_md}"}],
        temperature=0.5,
        max_tokens=500,
    )
    elapsed = time.time() - start
    logger.info(f"   {platform} completado en {elapsed:.1f}s")
    return response.content[0].text


# Release notes de ejemplo para Klimbook v2.4.0
SAMPLE_RELEASE_NOTES = """
## Klimbook v2.4.0

### Nuevas funcionalidades
- Sistema de logros: gana badges al completar rutas de diferentes grados
- Compartir rutas por QR code desde la app
- Modo offline mejorado: ahora guarda hasta 500 rutas sin conexión

### Correcciones
- Fix: crash al editar waypoints en rutas con más de 30 puntos
- Fix: las fotos se subían rotadas en algunos dispositivos Samsung
- Fix: notificaciones duplicadas al seguir a un usuario

### Mejoras
- Rendimiento del mapa mejorado un 40%
- Nueva animación de carga
"""

In [ ]:
async def step_format_all(notes_md: str) -> dict[str, str]:
    """Genera todas las versiones de release notes en paralelo."""
    platforms = ["github", "playstore", "appstore", "kofi"]

    logger.info("[Parallel] Lanzando 4 tareas en paralelo...")
    start = time.time()

    # asyncio.gather lanza todas las coroutines simultáneamente
    results = await asyncio.gather(
        *[format_for_platform(p, notes_md) for p in platforms]
    )

    elapsed = time.time() - start
    logger.info(f"[Parallel] Todas completadas en {elapsed:.1f}s (paralelo)")

    return dict(zip(platforms, results))


# Ejecutar y ver los resultados
all_versions = await step_format_all(SAMPLE_RELEASE_NOTES)

# Mostrar un preview de cada versión
for platform, content in all_versions.items():
    print(f"\n{'═' * 50}")
    print(f" {platform.upper()}")
    print(f"{'═' * 50}")
    print(content[:300])
    if len(content) > 300:
        print("...")

### `asyncio.as_completed` — procesar resultados conforme llegan

`asyncio.gather` espera a que **todas** las tareas terminen antes de devolver resultados. Pero a veces quieres **mostrar resultados progresivamente** — por ejemplo, en una UI donde cada plataforma se muestra conforme va quedando lista.

`asyncio.as_completed` devuelve un iterador de futuros que puedes awaitar en el **orden en que terminan**, no en el orden en que los lanzaste:

In [ ]:
async def step_format_all_progressive(notes_md: str) -> dict[str, str]:
    """Genera versiones y muestra cada una conforme termina."""
    platforms = ["github", "playstore", "appstore", "kofi"]

    # Crear tareas nombradas: necesitamos saber cuál tarea terminó
    # Para esto, envolvemos cada tarea para que devuelva también su nombre
    async def named_task(platform: str) -> tuple[str, str]:
        result = await format_for_platform(platform, notes_md)
        return platform, result

    tasks = [named_task(p) for p in platforms]
    results = {}

    logger.info("[Progressive] Lanzando tareas, mostrando conforme terminan...")
    for coro in asyncio.as_completed(tasks):
        platform, result = await coro
        results[platform] = result
        print(f"   {platform} listo ({len(results)}/{len(platforms)})")

    return results


# Ejecutar
progressive_results = await step_format_all_progressive(SAMPLE_RELEASE_NOTES)

### Rate Limiting con `asyncio.Semaphore`

En producción, los APIs tienen **límites de requests por minuto** (rate limits). Si tu plan permite 60 requests/minuto en la API de Anthropic y lanzas 100 en paralelo, vas a recibir errores `429 Too Many Requests`.

**`asyncio.Semaphore(n)`** es un candado que permite que **máximo `n` coroutines** se ejecuten simultáneamente. Las demás esperan su turno:

```
Semaphore(3) con 6 tareas:

Tiempo 0:   [Tarea1 🔵] [Tarea2 🔵] [Tarea3 🔵]  Tarea4 ⏳  Tarea5 ⏳  Tarea6 ⏳
Tiempo 1:   [Tarea1 ✅] [Tarea2 🔵] [Tarea3 🔵]  [Tarea4 🔵] Tarea5 ⏳  Tarea6 ⏳
Tiempo 2:   [Tarea2 ✅] [Tarea3 ✅] [Tarea4 🔵]  [Tarea5 🔵] [Tarea6 🔵]
```

In [ ]:
# Ejemplo: formatear release notes para MUCHAS plataformas con límite de concurrencia

# Límite: máximo 3 requests simultáneos a la API
API_SEMAPHORE = asyncio.Semaphore(3)

async def format_with_rate_limit(platform: str, notes: str) -> tuple[str, str]:
    """Formatea con límite de concurrencia para respetar rate limits."""
    async with API_SEMAPHORE:
        # Solo 3 de estas pueden ejecutarse al mismo tiempo
        # Las demás esperan hasta que una termine y libere el semáforo
        result = await format_for_platform(platform, notes)
        return platform, result


async def step_format_all_limited(notes_md: str) -> dict[str, str]:
    """Ejecuta con límite de concurrencia — seguro para producción."""
    # Imagina que necesitas generar para 8 plataformas/idiomas
    platforms = [
        "github", "playstore", "appstore", "kofi",
        # "playstore_es", "playstore_en", "appstore_es", "appstore_en",
    ]

    logger.info(f"[RateLimit] {len(platforms)} tareas con máx. 3 simultáneas")
    tasks = [format_with_rate_limit(p, notes_md) for p in platforms]
    results = await asyncio.gather(*tasks)

    return dict(results)


# Ejecutar
limited_results = await step_format_all_limited(SAMPLE_RELEASE_NOTES)
print(f"✅ {len(limited_results)} plataformas completadas con rate limiting")

### Error handling en paralelo con `return_exceptions=True`

¿Qué pasa si una de las 4 tareas falla? Por defecto, `asyncio.gather` **aborta todo** cuando una falla. Esto no es ideal — si la versión de Ko-fi falla, no quieres perder las versiones de GitHub, Play Store y App Store que ya terminaron.

Con `return_exceptions=True`, las excepciones se **devuelven como resultados** en lugar de propagarse. Así puedes manejar los errores individualmente:

In [ ]:
async def step_format_all_resilient(notes_md: str) -> dict[str, str]:
    """Genera versiones en paralelo — resiliente a fallos individuales."""
    platforms = ["github", "playstore", "appstore", "kofi"]

    results = await asyncio.gather(
        *[format_for_platform(p, notes_md) for p in platforms],
        return_exceptions=True,  # ← no abortar si una tarea falla
    )

    # Separar éxitos de errores
    output = {}
    for platform, result in zip(platforms, results):
        if isinstance(result, Exception):
            logger.error(f"  ❌ {platform} falló: {type(result).__name__}: {result}")
            output[platform] = None  # Marcar como fallido
        else:
            logger.info(f"  ✅ {platform} completado ({len(result)} chars)")
            output[platform] = result

    # Resumen
    successes = sum(1 for v in output.values() if v is not None)
    failures = sum(1 for v in output.values() if v is None)
    logger.info(f"[Resilient] {successes} éxitos, {failures} fallos")

    return output

---

## 4.3 Map-Reduce con LLMs

### El problema: documentos demasiado grandes

Imagina que quieres resumir **todas las reseñas de Klimbook** del último mes para un reporte ejecutivo. Son 500 reseñas, ~50,000 tokens en total. Aunque Claude soporta 200K tokens de contexto, procesar todo junto:

1. **Es caro** — estás pagando por 50,000 tokens de input en cada llamada.
2. **Puede perder detalles** — con inputs muy largos, el modelo puede "olvidar" información del medio.
3. **Podría exceder los límites** — con documentos aún más grandes.

### La solución: Map-Reduce

El patrón **Map-Reduce** (popularizado por Google/Hadoop) se adapta perfecto a LLMs:

```
500 reseñas de Klimbook (50,000 tokens)
    ↓
┌──────────────────┐
│    CHUNKING      │  ← Dividir en partes manejables (ej: 50 reseñas cada una)
└───────┬──────────┘
        │
        ├→ Chunk 1 (50 reseñas) → Haiku → Resumen 1
        ├→ Chunk 2 (50 reseñas) → Haiku → Resumen 2     ← MAP (en paralelo)
        ├→ Chunk 3 (50 reseñas) → Haiku → Resumen 3
        ├→ ...
        └→ Chunk 10            → Haiku → Resumen 10
                                   ↓
                          ┌──────────────────┐
                          │     REDUCE       │  ← Combinar los 10 resúmenes
                          └───────┬──────────┘
                                  ↓
                         Reporte ejecutivo final
```

Las tres fases:

| Fase | Qué hace | Modelo sugerido | ¿En paralelo? |
|------|----------|-----------------|---------------|
| **Chunking** | Divide el documento en partes | N/A (código Python) | N/A |
| **Map** | Procesa cada chunk independientemente | Haiku (barato, muchas llamadas) | ✅ Sí |
| **Reduce** | Combina los resultados parciales | Sonnet (calidad para resultado final) | ❌ No |

> **¿Por qué Haiku para Map y Sonnet para Reduce?** La fase Map es repetitiva y el input es pequeño — Haiku es suficiente y mucho más barato para N llamadas. La fase Reduce es una sola llamada donde necesitas calidad para el resultado final.

### Fase 1: Chunking — estrategias de división

No todos los textos se dividen igual. Aquí hay tres estrategias para diferentes situaciones:

| Estrategia | Cuándo usarla | Ejemplo |
|------------|---------------|---------|
| **Por tamaño fijo** | Texto sin estructura clara | Logs, texto plano |
| **Por párrafos** | Texto con separaciones naturales | Artículos, reseñas individuales |
| **Con overlap** | Cuando las ideas cruzan bordes | Documentación técnica, narrativas |

In [ ]:
def chunk_by_size(text: str, max_tokens: int = 500) -> list[str]:
    """Divide por tamaño fijo de tokens (aproximado).

    Aproximación: 1 token ≈ 4 chars en inglés, ≈ 3 chars en español.
    Para producción, usa tiktoken o la API de tokenización del modelo.
    """
    chars_per_chunk = max_tokens * 3
    chunks = []
    for i in range(0, len(text), chars_per_chunk):
        chunks.append(text[i : i + chars_per_chunk])
    return chunks


def chunk_by_paragraphs(text: str, max_tokens: int = 500) -> list[str]:
    """Divide por párrafos respetando límites naturales.

    Agrupa párrafos hasta llenar el chunk, sin cortar a mitad de párrafo.
    Ideal para colecciones de reseñas, artículos, etc.
    """
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) > max_tokens * 3:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            current_chunk += "\n\n" + para

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def chunk_with_overlap(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Divide con overlap para no perder contexto en los bordes.

    El overlap asegura que ideas que cruzan el borde de un chunk
    aparezcan completas en al menos uno de los dos chunks adyacentes.

    Ejemplo con chunk_size=100, overlap=20:
      Chunk 1: chars[0:300]
      Chunk 2: chars[240:540]    ← los chars 240-300 están en ambos
      Chunk 3: chars[480:780]
    """
    chars = chunk_size * 3
    overlap_chars = overlap * 3
    chunks = []

    start = 0
    while start < len(text):
        end = start + chars
        chunks.append(text[start:end])
        start = end - overlap_chars  # retrocede para crear overlap

    return chunks


# Demo: ver cómo se divide un texto de ejemplo
demo_text = "Reseña 1: Klimbook es genial.\n\n" * 20  # Simular 20 reseñas
chunks = chunk_by_paragraphs(demo_text, max_tokens=200)
print(f"Texto total: {len(demo_text)} chars")
print(f"Chunks generados: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {len(chunk)} chars, {chunk[:50]}...")

### ¿Por qué importa el overlap?

Sin overlap, si una idea empieza al final del chunk 3 y termina al inicio del chunk 4, se pierde en el resumen de ambos. Con **10-20% de overlap**, te aseguras de capturar ideas que cruzan los bordes.

> **Regla práctica**: usa 10% de overlap para texto general, 20% para texto técnico o denso donde las ideas son largas.

### Fase 2: Map — procesar cada chunk en paralelo

Cada chunk se procesa de forma **independiente** con Haiku (barato y rápido). Combinamos esto con `asyncio.Semaphore` para respetar rate limits:

In [ ]:
async def map_summarize(chunk: str, chunk_id: int, total: int) -> str:
    """Resume un chunk individual de reseñas de Klimbook."""
    response = await client.messages.create(
        model=MODELS["haiku"]["id"],
        system="""Eres un analista de producto. Resume estas reseñas de usuarios de Klimbook
(app de escalada) en máximo 3-4 oraciones. Enfócate en:
- Temas recurrentes (positivos y negativos)
- Features mencionados específicamente
- Sentimiento general del grupo""",
        messages=[{"role": "user", "content": chunk}],
        temperature=0.0,
        max_tokens=300,
    )
    logger.info(f"  [Map] Chunk {chunk_id + 1}/{total} procesado")
    return response.content[0].text


async def map_phase(chunks: list[str], max_concurrent: int = 5) -> list[str]:
    """Procesa todos los chunks en paralelo con límite de concurrencia."""
    semaphore = asyncio.Semaphore(max_concurrent)
    total = len(chunks)

    async def process_with_limit(chunk: str, idx: int) -> str:
        async with semaphore:
            return await map_summarize(chunk, idx, total)

    tasks = [process_with_limit(chunk, i) for i, chunk in enumerate(chunks)]
    return await asyncio.gather(*tasks)

### Fase 3: Reduce — combinar resultados parciales

La fase reduce toma **todos los resúmenes parciales** y los combina en un **documento final coherente**. Aquí usamos Sonnet porque necesitamos calidad — es el entregable final:

In [ ]:
async def reduce_phase(summaries: list[str]) -> str:
    """Combina los resúmenes parciales en un reporte ejecutivo final."""
    # Formateamos cada resumen con su número de sección
    combined = "\n\n---\n\n".join(
        f"**Grupo {i + 1} de reseñas:**\n{summary}"
        for i, summary in enumerate(summaries)
    )

    response = await client.messages.create(
        model=MODELS["sonnet"]["id"],
        system="""Eres un analista de producto senior. Combina estos resúmenes parciales
                de reseñas de Klimbook en un reporte ejecutivo coherente que incluya:

                1. **Resumen general**: sentimiento y satisfacción general
                2. **Top 3 cosas positivas**: lo que más valoran los usuarios
                3. **Top 3 problemas**: las quejas más frecuentes
                4. **Oportunidades**: features o mejoras que los usuarios piden
                5. **Recomendación**: una acción concreta para el equipo de producto

                Elimina redundancias entre secciones. Sé conciso pero completo.""",
        messages=[{"role": "user", "content": combined}],
        temperature=0.3,
        max_tokens=1500,
    )
    return response.content[0].text

### Pipeline completo: Chunk → Map → Reduce

Ahora juntamos todo en una sola función que orquesta las tres fases:

In [ ]:
async def map_reduce_summarize(text: str, chunk_size: int = 500, overlap: int = 50) -> str:
    """Pipeline completo de Map-Reduce para resumir documentos grandes.

    1. Chunking: divide el texto en partes manejables con overlap
    2. Map: resume cada chunk en paralelo con Haiku (barato)
    3. Reduce: combina resúmenes con Sonnet (calidad)
    """
    # Fase 1: Chunking
    chunks = chunk_with_overlap(text, chunk_size=chunk_size, overlap=overlap)
    logger.info(f"[MapReduce] Fase 1 - Chunking: {len(chunks)} chunks creados")

    # Fase 2: Map (paralelo con Haiku — barato y rápido)
    start = time.time()
    summaries = await map_phase(chunks, max_concurrent=5)
    map_time = time.time() - start
    logger.info(f"[MapReduce] Fase 2 - Map completado en {map_time:.1f}s: {len(summaries)} resúmenes")

    # Fase 3: Reduce (una sola llamada con Sonnet — calidad)
    start = time.time()
    final = await reduce_phase(summaries)
    reduce_time = time.time() - start
    logger.info(f"[MapReduce] Fase 3 - Reduce completado en {reduce_time:.1f}s")
    logger.info(f"[MapReduce] Total: {map_time + reduce_time:.1f}s")

    return final


# Simular reseñas de Klimbook para probar el pipeline
SAMPLE_REVIEWS = """
Reseña 1: Me encanta poder trackear mis rutas de boulder. La interfaz es muy limpia.

Reseña 2: La app se crashea cuando intento subir fotos en formato HEIC desde mi iPhone 15.

Reseña 3: Sería genial poder ver un mapa de calor con las zonas donde más escalo.

Reseña 4: El sistema de grados es confuso. ¿Por qué mezclan francés y Yosemite?

Reseña 5: Llevo 3 meses usando Klimbook y ya no uso ninguna otra app de escalada. 5 estrellas.

Reseña 6: El modo offline no funciona bien, pierdo mis logs cuando no hay WiFi en el rocódromo.

Reseña 7: Me gustaría poder exportar mis estadísticas a CSV para hacer mis propios análisis.

Reseña 8: La comunidad es lo mejor de la app. Encontré compañeros de escalada cerca de mi ciudad.

Reseña 9: Los filtros de búsqueda de rutas son muy limitados. Necesito filtrar por tipo de roca.

Reseña 10: Actualización reciente rompió las notificaciones push. Ya no me llegan.
""".strip()

# Ejecutar map-reduce
final_report = await map_reduce_summarize(SAMPLE_REVIEWS, chunk_size=200, overlap=20)
print("=" * 60)
print("REPORTE EJECUTIVO — Reseñas de Klimbook")
print("=" * 60)
print(final_report)

### ¿Cuándo usar Map-Reduce vs un solo prompt?

Claude soporta **200K tokens** de contexto, así que muchas veces un solo prompt es suficiente. Map-reduce añade complejidad y puede perder conexiones entre secciones distantes del documento.

| Criterio | Un solo prompt | Map-Reduce |
|----------|---------------|------------|
| **Tamaño del texto** | < 100K tokens | > 100K tokens o excede el límite |
| **Costo** | Pagas por todo el input una vez | Pagas por N llamadas pequeñas + 1 reduce |
| **Calidad** | Mejor para textos que necesitan visión global | Mejor para textos donde las secciones son independientes |
| **Velocidad** | Una sola llamada (puede ser lenta con input grande) | Map en paralelo puede ser más rápido |
| **Ejemplo** | Resumir un artículo de 5 páginas | Resumir 500 reseñas de usuarios |

> **Regla general**: empieza con un solo prompt. Si es demasiado lento, caro, o se pierde información, entonces implementa map-reduce.

---

## 4.4 Introducción a LangChain (referencia)

**No vamos a usar LangChain en este track**, pero es el framework más popular del ecosistema LLM y es importante entender qué es para poder evaluarlo después con criterio propio.

### ¿Qué es LangChain?

LangChain es un framework de Python/JS que abstrae las interacciones con LLMs. Proporciona componentes pre-construidos para tareas comunes:

| Componente | Qué hace | Equivalente en nuestro código |
|------------|----------|-------------------------------|
| **ChatModels** | Wrappers de APIs de LLMs | `client.messages.create()` |
| **PromptTemplates** | Templates con variables | f-strings o `.format()` |
| **OutputParsers** | Parsean el output del LLM | `json.loads()`, Pydantic |
| **Chains** | Combinan prompt + model + parser | Nuestras funciones `step_*` |
| **Agents** | LLMs que deciden qué tools usar | Lo veremos en la Unidad 3 |
| **Memory** | Persistencia de conversación | Manejar el array de `messages` |

### LCEL (LangChain Expression Language)

LangChain usa el operador `|` (pipe) para encadenar componentes, similar a Unix pipes. Así se ve un chain simple:

In [ ]:

# Requiere: pip install langchain langchain-anthropic

# --- Así se ve un chain en LangChain ---
# from langchain_anthropic import ChatAnthropic
# from langchain.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import JsonOutputParser

# prompt = ChatPromptTemplate.from_messages([
#     ("system", "Clasifica el mensaje en: bug_report, feature_request, question, feedback"),
#     ("user", "{message}"),
# ])
# model = ChatAnthropic(model="claude-3-5-haiku-20241022", temperature=0)
# parser = JsonOutputParser()

# # Componer con el operador |
# chain = prompt | model | parser

# # Usar:
# result = chain.invoke({"message": "La app se crashea al subir fotos"})

# --- Equivale a nuestro código: ---
# result = parser(model(prompt(input_data)))
# O en nuestro estilo: result = await step_classify("La app se crashea al subir fotos")

### ¿Cuándo usar LangChain y cuándo no?

| ✅ Usar LangChain cuando... | ❌ No usarlo cuando... |
|------------------------------|------------------------|
| Necesitas **prototyping rápido** con integraciones pre-hechas | Necesitas **control fino** del comportamiento |
| Quieres usar **vectorstores, document loaders, o tools** que ya existen | La abstracción te **estorba para debuggear** |
| El ecosistema te **ahorra tiempo significativo** | **Performance importa** (LangChain agrega overhead) |
| Trabajas en equipo y necesitas un estándar compartido | Estás **aprendiendo** (oculta los primitivos) |

### La postura de este track

Ya construiste routers, parallel chains, y map-reduce **con Python puro**. Ahora entiendes los primitivos. Si en el futuro decides usar LangChain, sabrás **exactamente** qué hace por debajo y podrás:

1. **Debuggear** cuando algo falle (y va a fallar).
2. **Decidir** cuándo la abstracción te ayuda y cuándo te estorba.
3. **Optimizar** porque entiendes el costo real de cada operación.

> *"No uses un framework hasta que entiendas qué problema resuelve. Y para entender el problema, primero resuélvelo sin framework."*

---

## Resumen de Conceptos Clave — Semana 4

### Patrones aprendidos

| Patrón | Cuándo usarlo | Herramienta clave |
|--------|---------------|-------------------|
| **Router Chain** | Input heterogéneo que necesita tratamiento diferenciado | Clasificador LLM + dict de handlers |
| **Parallel Chain** | Tareas independientes que pueden ejecutarse simultáneamente | `asyncio.gather()` |
| **Map-Reduce** | Documentos grandes que exceden la ventana o son muy caros | Chunking + `asyncio.gather()` + reduce |

### Conceptos técnicos

| Concepto | Descripción |
|----------|-------------|
| **Confidence threshold** | Pedir al clasificador un score de confianza y pedir clarificación si es bajo |
| **Assistant prefill** | Truco de agregar `{"role": "assistant", "content": "{"}` para forzar formato JSON |
| **`asyncio.gather`** | Ejecuta tareas en paralelo; el tiempo total = el del más lento |
| **`asyncio.as_completed`** | Procesa resultados conforme terminan, no en orden de lanzamiento |
| **`asyncio.Semaphore(n)`** | Limita a máximo `n` tareas simultáneas (rate limiting) |
| **`return_exceptions=True`** | Las excepciones se devuelven como resultados en lugar de abortar todo |
| **Chunking con overlap** | 10-20% de overlap para capturar ideas que cruzan bordes de chunks |

### Modelo mental para elegir qué patrón usar

```
¿Necesitas procesar inputs diferentes?
  → Sí: Router Chain (4.1)
  → No: ↓

¿Tienes tareas independientes?
  → Sí: Parallel Chain (4.2)
  → No: ↓

¿El documento es demasiado grande?
  → Sí: Map-Reduce (4.3)
  → No: Chain secuencial (lección anterior)
```
